In [1]:
import glob
import os

#set current working directory
os.chdir('/home/jovyan')

#list files in shared folder
files = glob.glob("/shared_space/Galveston_Bay/ERA5/*")
files

['/shared_space/Galveston_Bay/ERA5/ERA5-2019-2020_v3.nc']

In [5]:
import numpy as np
import xarray as xr

lat_new = np.arange(28.5, 30.0+0.5, 0.1)
lon_new = np.arange(-96.0, -94.0, 0.1)

ds = xr.open_dataset("/shared_space/Galveston_Bay/ERA5/ERA5-2018-2020.nc")
ds_highres = ds.interp(
    latitude=lat_new,
    longitude=lon_new,
    method="linear"  # or "nearest"
)


ds_highres['wind_speed'] = (ds_highres['u10']**2 + ds_highres['v10']**2)**0.5

ds_highres.to_netcdf("/shared_space/Galveston_Bay/ERA5/ERA5-2018-2020_rescaled.nc")


In [4]:
import xarray as xr

def open_clean(path):
    ds = xr.open_dataset(path)
    if "expver" in ds:
        ds = ds.drop("expver")
    return ds

ds1 = open_clean("/shared_space/Galveston_Bay/ERA5/uv_wind_2018-2020.nc")
ds2 = open_clean("/shared_space/Galveston_Bay/ERA5/ERA5_extra_2018-2020.nc")

ds2=ds2.drop('number')
# Combine along time (xarray auto-detects the dimension)
combined = xr.merge([ds1, ds2])
combined

for var in combined.data_vars:
    if 'coordinates' in combined[var].attrs:
        del combined[var].attrs['coordinates']
# Save to a new file
combined.to_netcdf("/shared_space/Galveston_Bay/ERA5/ERA5-2018-2020.nc")

In [2]:
import xarray as xr

def open_clean(path):
    ds = xr.open_dataset(path)
    if "expver" in ds:
        ds = ds.drop("expver")
    return ds

ds1 = open_clean("/shared_space/Galveston_Bay/ERA5/uv_wind_2018.nc")
ds2 = open_clean("/shared_space/Galveston_Bay/ERA5/uv_wind_2019.nc")
ds3 = open_clean("/shared_space/Galveston_Bay/ERA5/uv_wind_2020.nc")

# Combine along time (xarray auto-detects the dimension)
combined = xr.concat([ds1, ds2, ds3], dim="valid_time")

# Save to a new file
combined.to_netcdf("/shared_space/Galveston_Bay/ERA5/uv_wind_2018-2020.nc")